<a href="https://colab.research.google.com/github/Suganth1704/notebook/blob/main/6_fine_tuning/3_fine_tuning_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Roadmap
1. SFT dataset structure
Question → Correct Answer
2. Load the base model
3. Configure LoRA
4. Apply LoRA to the model
5. Inspect trainable parameters
6. Train the model
7. Save the adapter
8. Load the adapter with the base model
9. Compare before vs after

In [38]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [39]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes

| Library        | Purpose                    |
| -------------- | -------------------------- |
| `transformers` | Model and tokenizer        |
| `datasets`     | Training dataset           |
| `peft`         | LoRA                       |
| `trl`          | Supervised fine-tuning     |
| `accelerate`   | Training/device management |
| `bitsandbytes` | 4-bit quantization         |


In [40]:
import pandas as pd

import os
BASE_DIR = os.getcwd()
sample_data_path = os.path.join(BASE_DIR, "sample_data", "sample_data.csv")
df = pd.read_csv(sample_data_path)
df.head()

,instruction,input,output,category
0,How many annual leave days do employees receive?,NaN,Full-time employees receive 20 days of annual ...,Leave Policy
1,Can annual leave be carried forward?,NaN,Up to 5 unused leave days may be carried forwa...,Leave Policy
2,How do I apply for leave?,NaN,Submit your leave request through the HR porta...,Leave Policy
3,Is sick leave paid?,NaN,"Yes, employees receive up to 10 paid sick leav...",Leave Policy
4,Can I work from home?,NaN,Employees may work from home up to 3 days per ...,WFH


In [41]:
##Step 1 : Create a small dataset

from datasets import Dataset

# data = [
#     {
#         "instruction": "What is the capital of France?",
#         "output": "The capital of France is Paris."
#     },
#     {
#         "instruction": "What is the capital of Germany?",
#         "output": "The capital of Germany is Berlin."
#     },
#     {
#         "instruction": "What is the capital of Italy?",
#         "output": "The capital of Italy is Rome."
#     },
#     {
#         "instruction": "What is the capital of Japan?",
#         "output": "The capital of Japan is Tokyo."
#     },
#      {
#         "instruction": "Who is Suganth?",
#         "output": "Suganth is sudent from Chennai, Tamil Nadu."
#     },
#     {
#         "instruction": "What is the capital of India?",
#         "output": "The capital of India is New Delhi."
#     },
# ]

data = df.to_dict(orient="records")

dataset = Dataset.from_list(data)

print(dataset)


Dataset({
    features: ['instruction', 'input', 'output', 'category'],
    num_rows: 250
})


In [42]:
## Step 2: Format the dataset

def fromat_example(example):
  return {
      "text": f"""Question: {example['instruction']}\n\nAnswer: {example['output']}
      """
  }

formatted_dataset = dataset.map(fromat_example)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Question: How many annual leave days do employees receive?

Answer: Full-time employees receive 20 days of annual leave per calendar year.
      


In [43]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [44]:
import os
os.environ["HF_TOKEN"]=HF_TOKEN

In [45]:
## Step 3: Load the tokenizer

from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokens = tokenizer(
    formatted_dataset[0]["text"]
)

print(tokens["input_ids"])
print(len(tokens["input_ids"]))

[14582, 25, 2585, 1657, 9775, 5274, 2849, 653, 8256, 5258, 1939, 16141, 25, 8627, 7246, 8256, 5258, 220, 17, 15, 2849, 315, 9775, 5274, 817, 13168, 1042, 624, 981]
29


In [46]:
print(tokens)

{'input_ids': [14582, 25, 2585, 1657, 9775, 5274, 2849, 653, 8256, 5258, 1939, 16141, 25, 8627, 7246, 8256, 5258, 220, 17, 15, 2849, 315, 9775, 5274, 817, 13168, 1042, 624, 981], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


  The process is now:
  
    Text
      ↓
    Tokenizer
      ↓
    Token IDs
      ↓
    Model

In [47]:
##Padding

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [48]:
## Step 4: Load the base model

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

BitsAndBytesConfig:

How should the model weights be stored and computed using quantization?

Normally, model weights might use:

    FP32 → 32 bits per parameter
    FP16 → 16 bits per parameter
    INT8 → 8 bits per parameter
    INT4 → 4 bits per parameter

    ┌────────────────────────────┐
    │ Original Model Weights     │
    │                            │
    │        FP16                │
    └─────────────┬──────────────┘
                  │
                  ▼
          NF4 Quantization
                  │
                  ▼
              4-bit Storage
                  │
                  ▼
      ┌────────────────────────────┐
      │ Base Model                 │
      │                            │
      │ 4-bit NF4 weights          │ ❄️ Frozen
      │                            │
      └─────────────┬──────────────┘
                    │
                    ▼
            Transformer Layers
                    │
                    ├──────────────┐
                    │              │
                    ▼              ▼
              Base Output      LoRA Adapter
              (4-bit weights)  (FP16/BF16)
                    │              │
                    └──────┬───────┘
                          ▼
                        Combined
                          │
                          ▼
                        Loss
                          │
                          ▼
                  Update LoRA only

In [49]:
## Step 5: Configure LoRA

from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

In [50]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_params = 0

    for _, param in model.named_parameters():
        all_params += param.numel()

        if param.requires_grad:
            trainable_params += param.numel()

    print(f"Trainable params: {trainable_params:,}")
    print(f"All params: {all_params:,}")
    print(
        f"Trainable %: "
        f"{100 * trainable_params / all_params:.4f}%"
    )

print_trainable_parameters(model)

Trainable params: 233,518,592
All params: 888,616,448
Trainable %: 26.2789%


In [51]:
# for name, param in model.named_parameters():
#     if param.requires_grad:
#         print(name, param.shape)

In [52]:
### Step 6: prepare the model correctly for k-bit training.

from peft import prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

In [53]:
count = 0
for name, param in model.named_parameters():
    if param.requires_grad:
      count += 1
      #print(name, param.shape)
print(count)

224


In [54]:
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [55]:
##let's inspect one actual LoRA layer.
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)
        print(module)
        break


base_model.model.model.layers.0.self_attn.q_proj
lora.Linear4bit(
  (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=1536, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=1536, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)


In [56]:
import os
os.getcwd() ## to check the current dir in google colab

'/content'

In [57]:
## Step 6: Configure the training arguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "/content/sample_data/qwen_lora",
    num_train_epochs = 5,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-4,
    fp16 = False,
    bf16=True,
    logging_steps = 1,
    save_strategy = "epoch",
    report_to="none"
)

In [58]:
## Step 7: Create the Trainer

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

In [59]:
## Start training

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,4.812937
2,3.876242
3,4.008730
4,3.784377
5,4.024796
6,4.143421
7,3.143385
8,4.055960
9,3.712450
10,3.269620


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=315, training_loss=0.5631275222415015, metrics={'train_runtime': 564.8738, 'train_samples_per_second': 2.213, 'train_steps_per_second': 0.558, 'total_flos': 268502150937600.0, 'train_loss': 0.5631275222415015, 'epoch': 5.0})

In [60]:
print(f"Is BFloat16 supported on this GPU? {torch.cuda.is_bf16_supported()}")

Is BFloat16 supported on this GPU? True


In [61]:
## Testing

adapter_path  = "/content/sample_data/qwen_lora_adapter"

trainer.save_model(adapter_path)

In [62]:
tokenizer.save_pretrained(adapter_path)

('/content/sample_data/qwen_lora_adapter/tokenizer_config.json',
 '/content/sample_data/qwen_lora_adapter/chat_template.jinja',
 '/content/sample_data/qwen_lora_adapter/tokenizer.json')

In [63]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [64]:
fine_tuned_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.L

In [65]:
def generate_response(model, question):
    prompt = f"""Question: {question}\n\nAnswer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    # Decode the response, starting after the input prompt to get only the generated answer
    response = tokenizer.decode(
        outputs[0][len(inputs["input_ids"][0]):],
        skip_special_tokens=True
    ).strip()

    return response

In [66]:
print(
    generate_response(
        fine_tuned_model,
        "Capital of india"
    )
)

New Delhi


In [77]:
fresh_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [78]:
query = "Can I work from home?"

print("BASE MODEL:")
print(generate_response(fresh_base_model, query))

print("FINE TUNED MODEL:")
print(generate_response(fine_tuned_model, query))

BASE MODEL:
Yes, you can work from home. Many companies offer remote work options and it is becoming more common for employees to work from home. However, it's important to note that not all jobs are suitable for remote work and some may require a physical presence
FINE TUNED MODEL:
Employees may work from home up to 3 days per week with manager approval.
